# **SPARK ON YARN - TIPE 1: PRAPEMROSESAN TERDISTRIBUSI & PELATIHAN TERPUSAT GPU**
## **SKENARIO IRISAN 3 KELAS (NEVUS, MELANOMA, SEBORRHEIC KERATOSIS)**

Notebook ini mengimplementasikan arsitektur hybrid komputasi data besar:
1. **PySpark on YARN:** Digunakan untuk ingesti data dari HDFS dan prapemrosesan citra terdistribusi secara paralel di seluruh worker node.
2. **Centralized GPU Training:** Partisi data yang telah diproses dikumpulkan ke driver untuk melatih arsitektur deep learning ResNet-50 dengan modul Triplet Attention (Baseline AK85) menggunakan akselerasi GPU.

Alur kerja dirancang dengan metodologi CRISP-DM yang ketat dan transparan:
- **Fase 1: Setup Lingkungan Komputasi** (Instalasi Java 11, Hadoop 3.3.6, Spark 3.5.5, SSH)
- **Fase 2: Konfigurasi HDFS** (Penyimpanan data terdistribusi pseudo-distributed)
- **Fase 3: Konfigurasi YARN** (Manajemen resource CPU & RAM untuk cluster worker)
- **Fase 4: Data Understanding & Ingesti HDFS** (Eksplorasi data, audit ukuran, blok HDFS, visualisasi citra asli)
- **Fase 5: Pipeline Preprocessing Terdistribusi** (PySpark RDD, resize 224x224, normalisasi [0, 1], bukti histogram)
- **Fase 6: Data Preparation (Splitting) & Arsitektur Model** (Split Train/Val/Test tanpa kebocoran data, konstruksi ResNet50 + Triplet Attention)
- **Fase 7: Pelatihan Terpusat GPU & Evaluasi Komprehensif** (Training GPU, kurva evaluasi, confusion matrix, classification report, logging)

# **FASE 1 - Set up Env**
1. Deteksi Lingkungan & Mount Storage (Mendukung Kaggle & Google Colab)
2. Konfigurasi Skenario dan Path Kerja
3. Generate dan Jalankan `setup_env.sh` (Java 11, Hadoop, Spark, SSH)
4. Set Environment Variables
5. Verifikasi Instalasi Dasar

### **1. Deteksi Lingkungan & Mount Storage**
Notebook secara otomatis mendeteksi apakah dijalankan di Google Colab atau Kaggle. Jika di Colab, Google Drive dimuat untuk penyimpanan cache.

In [ ]:
from pathlib import Path
import os
import sys

# Deteksi lingkungan eksekusi secara otomatis
if Path("/kaggle/working").exists():
    ENV_NAME = "kaggle"
    INSTALL_DIR = Path("/kaggle/working")
    PROJECT_DIR = INSTALL_DIR / "riset_kanker_kulit"
    print("Terdeteksi lingkungan: Kaggle Notebook")
else:
    ENV_NAME = "colab"
    INSTALL_DIR = Path("/content")
    PROJECT_DIR = Path("/content/drive/MyDrive/riset_kanker_kulit")
    print("Terdeteksi lingkungan: Google Colab")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Peringatan saat mount drive:", e)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Direktori kerja utama: {PROJECT_DIR}")

### **2. Konfigurasi Skenario dan Path**
Pengaturan skenario eksperimen meliputi jumlah worker YARN, jumlah partisi RDD, serta jumlah epoch pelatihan.

In [ ]:
# Konfigurasi skenario eksperimen
NUM_WORKERS = 2
NUM_PARTITIONS = 8
EPOCHS = 15
SCENARIO_TYPE = "irisan"

SCENARIO_NAME = f"{SCENARIO_TYPE}_{NUM_WORKERS}W{NUM_PARTITIONS}P"

# Versi komponen sistem
HADOOP_VERSION = "3.3.6"
SPARK_VERSION = "3.5.5"
SPARK_HADOOP_PROFILE = "hadoop3"

SSH_PORT = 2222

# Path instalasi lokal (ephemeral per sesi)
HADOOP_HOME = INSTALL_DIR / "hadoop"
SPARK_HOME = INSTALL_DIR / "spark"
HADOOP_CONF_DIR = HADOOP_HOME / "etc" / "hadoop"

DATA_DIR = INSTALL_DIR / "hadoop_data"
NAMENODE_DIR = DATA_DIR / "namenode"
DATANODE_DIR = DATA_DIR / "datanode"

HDFS_WORKDIR = f"/user/skincancer/{SCENARIO_TYPE}"

# Direktori cache dan script
CACHE_DIR = PROJECT_DIR / "cache"
SCRIPT_DIR = PROJECT_DIR / "scripts"
SETUP_SCRIPT_PATH = SCRIPT_DIR / "setup_env.sh"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Skenario: {SCENARIO_NAME}")
print(f"Hadoop Home: {HADOOP_HOME}")
print(f"Spark Home: {SPARK_HOME}")
print(f"HDFS Workdir: {HDFS_WORKDIR}")

### **3. Generate dan Jalankan `setup_env.sh`**
Skrip bash ini bersifat idempoten dan mengunduh Java 11, Hadoop 3.3.6, Spark 3.5.5, serta mengonfigurasi SSH tanpa password untuk komunikasi master-worker di localhost.

In [ ]:
SETUP_SCRIPT = f"""#!/bin/bash
set -e

HADOOP_VERSION="{HADOOP_VERSION}"
SPARK_VERSION="{SPARK_VERSION}"
SPARK_HADOOP_PROFILE="{SPARK_HADOOP_PROFILE}"

INSTALL_DIR="{INSTALL_DIR}"
HADOOP_HOME="{HADOOP_HOME}"
SPARK_HOME="{SPARK_HOME}"
CACHE_DIR="{CACHE_DIR}"

HADOOP_ARCHIVE="hadoop-${{HADOOP_VERSION}}.tar.gz"
SPARK_ARCHIVE="spark-${{SPARK_VERSION}}-bin-${{SPARK_HADOOP_PROFILE}}.tgz"

HADOOP_URL="https://archive.apache.org/dist/hadoop/common/hadoop-${{HADOOP_VERSION}}/${{HADOOP_ARCHIVE}}"
SPARK_URL="https://archive.apache.org/dist/spark/spark-${{SPARK_VERSION}}/${{SPARK_ARCHIVE}}"

mkdir -p "${{CACHE_DIR}}"

fetch_archive() {{
    local archive_name="$1"
    local url="$2"
    local local_path="${{INSTALL_DIR}}/${{archive_name}}"
    local cache_path="${{CACHE_DIR}}/${{archive_name}}"

    if [ -f "${{local_path}}" ]; then
        return 0
    fi

    if [ -f "${{cache_path}}" ]; then
        echo "Menggunakan cache arsip dari storage persisten: ${{archive_name}}"
        cp "${{cache_path}}" "${{local_path}}"
    else
        echo "Mengunduh ${{archive_name}}..."
        wget -q "${{url}}" -O "${{local_path}}"
        if [ -d "${{CACHE_DIR}}" ]; then
            cp "${{local_path}}" "${{cache_path}}" || true
        fi
    fi
}}

echo "=== 1. Memeriksa Java 11 ==="
dpkg -s openjdk-11-jdk-headless >/dev/null 2>&1 || {{
    apt-get update -qq
    apt-get install -y -qq openjdk-11-jdk-headless
}}

echo "=== 2. Memeriksa Hadoop ${{HADOOP_VERSION}} ==="
if [ ! -d "${{HADOOP_HOME}}" ]; then
    fetch_archive "${{HADOOP_ARCHIVE}}" "${{HADOOP_URL}}"
    tar -xzf "${{INSTALL_DIR}}/${{HADOOP_ARCHIVE}}" -C "${{INSTALL_DIR}}"
    mv "${{INSTALL_DIR}}/hadoop-${{HADOOP_VERSION}}" "${{HADOOP_HOME}}"
fi

echo "=== 3. Memeriksa Spark ${{SPARK_VERSION}} ==="
if [ ! -d "${{SPARK_HOME}}" ]; then
    fetch_archive "${{SPARK_ARCHIVE}}" "${{SPARK_URL}}"
    tar -xzf "${{INSTALL_DIR}}/${{SPARK_ARCHIVE}}" -C "${{INSTALL_DIR}}"
    mv "${{INSTALL_DIR}}/spark-${{SPARK_VERSION}}-bin-${{SPARK_HADOOP_PROFILE}}" "${{SPARK_HOME}}"
fi

echo "=== 4. Konfigurasi SSH Localhost ==="
dpkg -s openssh-server >/dev/null 2>&1 || apt-get install -y -qq openssh-server openssh-client
mkdir -p /var/run/sshd ~/.ssh

ssh-keygen -A

[ -f ~/.ssh/id_rsa ] || ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa -q
touch ~/.ssh/authorized_keys
grep -qxF "$(cat ~/.ssh/id_rsa.pub)" ~/.ssh/authorized_keys || cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 700 ~/.ssh
chmod 600 ~/.ssh/authorized_keys

# Pastikan port SSH sesuai SSH_PORT
sed -i 's/#Port 22/Port 2222/g' /etc/ssh/sshd_config || true
sed -i 's/Port 22/Port 2222/g' /etc/ssh/sshd_config || true

service ssh restart

echo "Setup lingkungan selesai: Java 11, Hadoop, Spark, dan SSH siap."
"""

SETUP_SCRIPT_PATH.write_text(SETUP_SCRIPT)
SETUP_SCRIPT_PATH.chmod(0o755)
print(f"Skrip setup berhasil dibuat: {SETUP_SCRIPT_PATH}")

In [ ]:
!bash "{SETUP_SCRIPT_PATH}" 

### **4. Set Environment Variables**
Menentukan path `JAVA_HOME`, `HADOOP_HOME`, dan `SPARK_HOME` secara dinamis pada environment runtime Python.

In [ ]:
import os
import subprocess

def resolve_java_home() -> str:
    candidate = Path("/usr/lib/jvm/java-11-openjdk-amd64")
    if candidate.exists():
        return str(candidate)

    alternatives = subprocess.run(
        ["update-alternatives", "--list", "java"], capture_output=True, text=True
    ).stdout.splitlines()
    for path in alternatives:
        if "java-11" in path:
            return str(Path(path).parent.parent)

    which_java = subprocess.run(
        ["which", "java"], capture_output=True, text=True, check=True
    ).stdout.strip()
    real_java = subprocess.run(
        ["readlink", "-f", which_java], capture_output=True, text=True, check=True
    ).stdout.strip()
    return str(Path(real_java).parent.parent)

os.environ["JAVA_HOME"] = resolve_java_home()
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
os.environ["SPARK_HOME"] = str(SPARK_HOME)
os.environ["HADOOP_CONF_DIR"] = str(HADOOP_CONF_DIR)
os.environ["LD_LIBRARY_PATH"] = str(HADOOP_HOME / "lib" / "native")

os.environ["PATH"] = ":".join([
    f"{os.environ['JAVA_HOME']}/bin",
    f"{os.environ['HADOOP_HOME']}/bin",
    f"{os.environ['HADOOP_HOME']}/sbin",
    f"{os.environ['SPARK_HOME']}/bin",
    f"{os.environ['SPARK_HOME']}/sbin",
    os.environ["PATH"],
])

print(f"JAVA_HOME   = {os.environ['JAVA_HOME']}")
print(f"HADOOP_HOME = {os.environ['HADOOP_HOME']}")
print(f"SPARK_HOME  = {os.environ['SPARK_HOME']}")

### **5. Verifikasi Instalasi Dasar**
Memastikan binary Java, Hadoop, Spark, dan konektivitas SSH localhost berjalan normal.

In [ ]:
!java -version
!$HADOOP_HOME/bin/hadoop version
!$SPARK_HOME/bin/spark-submit --version
!ssh -o StrictHostKeyChecking=no -p {SSH_PORT} localhost "echo SSH connection OK" 

# **FASE 2 - Konfigurasi HDFS**
6. Menulis `core-site.xml` (Pengalamatan default NameNode)
7. Menulis `hdfs-site.xml` & Daftarkan `JAVA_HOME` ke `hadoop-env.sh`
8. Format NameNode (Idempotent)
9. Menjalankan HDFS Daemons & Verifikasi JPS
10. Membuat Direktori Kerja HDFS

### **6. `core-site.xml`**
Mengarahkan default filesystem HDFS ke port 9000 localhost.

In [ ]:
CORE_SITE = """<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>
"""

core_site_path = HADOOP_CONF_DIR / "core-site.xml"
core_site_path.write_text(CORE_SITE)
print(f"Tersimpan: {core_site_path}")

### **7. `hdfs-site.xml` dan `hadoop-env.sh`**
Mengonfigurasi direktori penyimpanan fisik blok data NameNode & DataNode, serta mematikan pengecekan permission agar tidak terjadi kendala akses.

In [ ]:
NAMENODE_DIR.mkdir(parents=True, exist_ok=True)
DATANODE_DIR.mkdir(parents=True, exist_ok=True)

HDFS_SITE = f"""<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file://{NAMENODE_DIR}</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file://{DATANODE_DIR}</value>
    </property>
    <property>
        <name>dfs.permissions.enabled</name>
        <value>false</value>
    </property>
    <property>
        <name>dfs.namenode.secondary.http-address</name>
        <value>localhost:9868</value>
    </property>
</configuration>
"""

hdfs_site_path = HADOOP_CONF_DIR / "hdfs-site.xml"
hdfs_site_path.write_text(HDFS_SITE)
print(f"Tersimpan: {hdfs_site_path}")

hadoop_env_path = HADOOP_CONF_DIR / "hadoop-env.sh"

env_exports = {
    "JAVA_HOME": os.environ["JAVA_HOME"],
    "HADOOP_SSH_OPTS": f"-p {SSH_PORT}",
    "HDFS_NAMENODE_USER": "root",
    "HDFS_DATANODE_USER": "root",
    "HDFS_SECONDARYNAMENODE_USER": "root",
    "YARN_RESOURCEMANAGER_USER": "root",
    "YARN_NODEMANAGER_USER": "root",
}

lines = [
    line for line in hadoop_env_path.read_text().splitlines()
    if not any(line.strip().startswith(f"export {key}=") for key in env_exports)
]
for key, value in env_exports.items():
    lines.append(f'export {key}="{value}"')

hadoop_env_path.write_text("\n".join(lines) + "\n")
print("hadoop-env.sh berhasil diperbarui.")

### **8. Format NameNode**
Memformat filesystem NameNode jika belum pernah diinisialisasi pada sesi ini.

In [ ]:
version_file = NAMENODE_DIR / "current" / "VERSION"

if version_file.exists():
    print("NameNode sudah pernah diformat sebelumnya, langkah ini dilewati.")
else:
    print("Memformat NameNode...")
    result = subprocess.run(
        [str(HADOOP_HOME / "bin" / "hdfs"), "namenode", "-format", "-force"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Format NameNode gagal")
    print("NameNode berhasil diformat.")

### **9. Start HDFS dan Verifikasi**
Menjalankan daemon NameNode, DataNode, dan SecondaryNameNode serta memvalidasi ketersediaannya lewat Java Process Status (JPS).

In [ ]:
!$HADOOP_HOME/sbin/start-dfs.sh

In [ ]:
def check_jps(expected_processes):
    output = subprocess.run(["jps"], capture_output=True, text=True).stdout
    print("Daftar proses Java (jps):")
    print(output)
    all_ok = True
    for name in expected_processes:
        status = "BERJALAN (OK)" if name in output else "TIDAK DITEMUKAN (MISSING)"
        print(f"  - {name}: {status}")
        if name not in output:
            all_ok = False
    return all_ok

check_jps(["NameNode", "DataNode"])

### **10. Buat Direktori Kerja di HDFS**
Membuat path kerja HDFS dan memeriksa kapasitas cluster.

In [ ]:
!hdfs dfs -mkdir -p {HDFS_WORKDIR}
!hdfs dfs -ls -R /user/skincancer
!hdfs dfsadmin -report

# **FASE 3 - Konfigurasi YARN**
11. Cek Kapasitas Aktual Lingkungan (CPU & RAM)
12. Hitung Alokasi Resource YARN dan Executor Dinamis
13. Menulis `yarn-site.xml`
14. Menulis `mapred-site.xml`
15. Menjalankan YARN Daemons & Verifikasi
16. Stage Spark Jars ke HDFS (`spark-libs.zip`)
17. Menulis `spark-defaults.conf`
18. Verifikasi Submit Job ke YARN (`SparkPi`)

### **11. Cek Kapasitas Aktual Lingkungan**
Memeriksa jumlah core CPU fisik dan memori RAM aktual yang tersedia pada container runtime.

In [ ]:
!nproc
!free -h

### **12. Hitung Alokasi Resource YARN dan Executor**
Mengalokasikan memori NodeManager dan container executor secara dinamis agar aman dari pembatalan memori virtual oleh YARN.

In [ ]:
import subprocess

cpu_count = int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip())
free_lines = subprocess.run(["free", "-m"], capture_output=True, text=True).stdout.splitlines()
total_mem_mb = int(free_lines[1].split()[1])

# Sisakan RAM 2GB untuk OS, kernel Jupyter, dan proses GPU/Driver
RESERVED_MEM_MB = 2048
YARN_NM_MEMORY_MB = max(total_mem_mb - RESERVED_MEM_MB, 2048)

MAX_WORKERS_PLANNED = max(NUM_WORKERS, 4)
YARN_NM_VCORES = max(cpu_count, MAX_WORKERS_PLANNED)

container_memory_mb = YARN_NM_MEMORY_MB // MAX_WORKERS_PLANNED
EXECUTOR_MEMORY_OVERHEAD_MB = max(384, int(container_memory_mb * 0.1))
EXECUTOR_MEMORY_MB = max(container_memory_mb - EXECUTOR_MEMORY_OVERHEAD_MB, 512)
EXECUTOR_CORES = 1

print(f"CPU Fisik: {cpu_count}, RAM Total: {total_mem_mb} MB")
print(f"YARN NodeManager: {YARN_NM_MEMORY_MB} MB, {YARN_NM_VCORES} vcores")
print(f"Alokasi per Executor: {EXECUTOR_MEMORY_MB} MB (+ {EXECUTOR_MEMORY_OVERHEAD_MB} MB overhead), {EXECUTOR_CORES} core")

### **13. `yarn-site.xml`**
Konfigurasi NodeManager dengan menonaktifkan pengecekan virtual memory (`vmem-check-enabled = false`) agar container tidak dimatikan paksa di lingkungan virtual.

In [ ]:
YARN_SITE = f"""<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>yarn.nodemanager.aux-services</name>
        <value>mapreduce_shuffle</value>
    </property>
    <property>
        <name>yarn.nodemanager.aux-services.mapreduce.shuffle.class</name>
        <value>org.apache.hadoop.mapred.ShuffleHandler</value>
    </property>
    <property>
        <name>yarn.nodemanager.resource.memory-mb</name>
        <value>{YARN_NM_MEMORY_MB}</value>
    </property>
    <property>
        <name>yarn.nodemanager.resource.cpu-vcores</name>
        <value>{YARN_NM_VCORES}</value>
    </property>
    <property>
        <name>yarn.nodemanager.vmem-check-enabled</name>
        <value>false</value>
    </property>
</configuration>
"""

yarn_site_path = HADOOP_CONF_DIR / "yarn-site.xml"
yarn_site_path.write_text(YARN_SITE)
print(f"Tersimpan: {yarn_site_path}")

### **14. `mapred-site.xml`**
Mengarahkan framework eksekusi MapReduce ke YARN.

In [ ]:
MAPRED_SITE = """<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>
</configuration>
"""

mapred_site_path = HADOOP_CONF_DIR / "mapred-site.xml"
mapred_site_path.write_text(MAPRED_SITE)
print(f"Tersimpan: {mapred_site_path}")

### **15. Start YARN dan Verifikasi**
Menjalankan ResourceManager dan NodeManager, lalu memvalidasi status proses Java.

In [ ]:
!$HADOOP_HOME/sbin/start-yarn.sh

In [ ]:
check_jps(["ResourceManager", "NodeManager"])

### **16. Stage Spark Jars ke HDFS**
Mengemas seluruh file JAR Spark ke dalam arsip zip dan menyimpannya di HDFS agar executor YARN dapat mendistribusikannya secara instan tanpa menduplikasi file via jaringan.

In [ ]:
import shutil

SPARK_ARCHIVE_HDFS_PATH = "/user/spark/spark-libs.zip"
spark_libs_zip = INSTALL_DIR / "spark-libs.zip"

if not spark_libs_zip.exists():
    print("Mengemas pustaka JAR Spark menjadi file zip...")
    shutil.make_archive(str(spark_libs_zip.with_suffix("")), "zip", str(SPARK_HOME / "jars"))

subprocess.run(["hdfs", "dfs", "-mkdir", "-p", "/user/spark"], check=True)
subprocess.run(["hdfs", "dfs", "-put", "-f", str(spark_libs_zip), SPARK_ARCHIVE_HDFS_PATH], check=True)
print(f"Pustaka Spark terpasang di HDFS: hdfs://localhost:9000{SPARK_ARCHIVE_HDFS_PATH}")

### **17. `spark-defaults.conf`**
Menentukan konfigurasi default PySpark saat terhubung ke master YARN.

In [ ]:
SPARK_DEFAULTS = f"""spark.master                     yarn
spark.submit.deployMode          client
spark.yarn.archive               hdfs://localhost:9000{SPARK_ARCHIVE_HDFS_PATH}
spark.executor.instances         {NUM_WORKERS}
spark.executor.cores             {EXECUTOR_CORES}
spark.executor.memory            {EXECUTOR_MEMORY_MB}m
spark.executor.memoryOverhead    {EXECUTOR_MEMORY_OVERHEAD_MB}m
spark.driver.memory              2g
"""

spark_defaults_path = SPARK_HOME / "conf" / "spark-defaults.conf"
spark_defaults_path.write_text(SPARK_DEFAULTS)
print(f"Tersimpan: {spark_defaults_path}")
print(SPARK_DEFAULTS)

### **18. Verifikasi: Submit Job ke YARN**
Menguji apakah klaster YARN dapat mengeksekusi job SparkPi dengan benar.

In [ ]:
!$SPARK_HOME/bin/spark-submit \
  --master yarn \
  --deploy-mode client \
  --class org.apache.spark.examples.SparkPi \
  $SPARK_HOME/examples/jars/spark-examples_*.jar 10

# **FASE 4 - Dataset (Data Understanding & Ingesti HDFS)**
Sesuai metodologi CRISP-DM, pada tahap **Data Understanding** data dipahami secara mendalam tanpa mendahului proses pelatihan atau prapemrosesan:
19. Konfigurasi Dataset dan Direktori Kelas
20. Persiapan Dataset Lokal (Mendukung data Kaggle Input / Download otomatis)
21. Cek Struktur Folder Data Bersih
22. Upload Dataset ke HDFS (`hdfs dfs -put`)
23. Verifikasi Dataset di HDFS (`hdfs dfs -ls -R`, `hdfs dfs -du -h`)
24. Ringkasan Jumlah File, Ukuran, dan Rasio Imbalance per Kelas
24.1. Analisis Distribusi Blok HDFS
24.2. Visualisasi Grafik Distribusi Data
24.3. Visualisasi Contoh Sampel Citra Mentah per Kelas

### **19. Konfigurasi Dataset dan Direktori Kelas**
Menentukan daftar kelas target dan direktori lokal untuk pengorganisasian citra sebelum diunggah ke HDFS.

In [ ]:
CLASS_NAMES = ["nevus", "melanoma", "seborrheic_keratosis"]
NUM_CLASSES = len(CLASS_NAMES)

DATASET_LOCAL_DIR = INSTALL_DIR / "dataset_raw"
MERGED_DATASET_DIR = INSTALL_DIR / f"dataset_{SCENARIO_TYPE}"

DATASET_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

for c in CLASS_NAMES:
    (MERGED_DATASET_DIR / c).mkdir(parents=True, exist_ok=True)

print(f"Kelas target (3 kelas): {CLASS_NAMES}")
print(f"Direktori lokal dataset: {MERGED_DATASET_DIR}")

### **20. Persiapan Dataset Lokal Menggunakan Master CSV**
Dataset citra disinkronkan langsung berdasarkan file manifest master hasil deduplikasi dan harmonisasi label medis yang telah kita buat sebelumnya:
- Skenario Biner: `dataset_binary_final.csv` (33.552 citra terverifikasi)
- Skenario Irisan: `dataset_irisan_multiclass_final.csv` (22.051 citra terverifikasi)
Tidak ada pemindaian ulang dari nol ataupun duplikasi kerja. File CSV master dibaca langsung sebagai ground truth untuk pengelompokan citra ke HDFS.

In [ ]:
import pandas as pd
import shutil

CSV_FILENAME = "dataset_irisan_multiclass_final.csv"

search_dirs = [
    PROJECT_DIR / "Dataset",
    PROJECT_DIR,
    Path("Dataset"),
    Path("/kaggle/input"),
    Path("/content/drive/MyDrive/riset_kanker_kulit/Dataset"),
    Path("/content/drive/MyDrive/riset_kanker_kulit"),
]

csv_path = None
for d in search_dirs:
    if not d.exists():
        continue
    candidate = d / CSV_FILENAME
    if candidate.exists():
        csv_path = candidate
        break
    matches = list(d.rglob(CSV_FILENAME))
    if matches:
        csv_path = matches[0]
        break

if csv_path is None:
    raise FileNotFoundError(f"File manifest master {CSV_FILENAME} tidak ditemukan! Pastikan file CSV diunggah.")

df_master = pd.read_csv(csv_path)
print(f"Berhasil memuat file manifest master: {csv_path.name}")
print(f"Total citra terdaftar: {len(df_master)} baris")

label_map = {"NV": "nevus", "MEL": "melanoma", "BKL": "seborrheic_keratosis"}
df_master["target_folder"] = df_master["harmonized_label"].map(label_map)
print("Distribusi kelas pada CSV master:")
print(df_master["target_folder"].value_counts())

existing_count = sum(len(list((MERGED_DATASET_DIR / c).glob("*.*"))) for c in CLASS_NAMES)
if existing_count >= len(df_master):
    print(f"Direktori lokal sudah terisi lengkap ({existing_count} citra), sinkronisasi dilewati.")
else:
    print("Menyinkronkan citra fisik ke folder kelas berdasarkan CSV master...")
    image_search_roots = [
        csv_path.parent,
        PROJECT_DIR / "Dataset",
        Path("Dataset"),
        Path("/kaggle/input"),
        Path("/content/drive/MyDrive/riset_kanker_kulit/Dataset"),
    ]

    copied = 0
    missing = 0
    for idx, row in df_master.iterrows():
        img_id = str(row["image_id"])
        folder_cls = row["target_folder"]
        if folder_cls not in CLASS_NAMES:
            continue

        dest_file = MERGED_DATASET_DIR / folder_cls / f"{img_id}.jpg"
        if dest_file.exists():
            continue

        src_path = Path(str(row["filepath"]))
        if not src_path.exists():
            found_src = None
            norm_p = str(row["filepath"]).replace("", "/")
            if "Dataset/" in norm_p:
                rel_p = norm_p.split("Dataset/")[-1]
                for root in image_search_roots:
                    if (root / rel_p).exists():
                        found_src = root / rel_p
                        break
            if not found_src:
                for root in image_search_roots:
                    if (root / f"{img_id}.jpg").exists():
                        found_src = root / f"{img_id}.jpg"
                        break
            src_path = found_src

        if src_path and src_path.exists():
            try:
                dest_file.symlink_to(src_path.resolve())
            except Exception:
                shutil.copy2(src_path, dest_file)
            copied += 1
        else:
            missing += 1

    print(f"Sinkronisasi selesai: {copied} citra disinkronkan ke folder kelas.")
    if missing > 0:
        print(f"Catatan: {missing} citra tidak ditemukan di direktori pencarian lokal.")


### **21. Extract Dataset dan Cek Struktur Folder**
Memeriksa struktur folder dataset lokal untuk memastikan setiap kelas terorganisir dengan benar.

In [ ]:
for c in CLASS_NAMES:
    folder = MERGED_DATASET_DIR / c
    count = len(list(folder.glob("*.*")))
    print(f"Folder kelas '{c}': {count} citra")

### **22. Upload Dataset ke HDFS**
Menyalin struktur direktori citra lokal ke dalam filesystem HDFS (`/user/skincancer/{scenario_type}/`).

In [ ]:
class_folders = [MERGED_DATASET_DIR / name for name in CLASS_NAMES]

existing_hdfs = subprocess.run(
    ["hdfs", "dfs", "-ls", HDFS_WORKDIR], capture_output=True, text=True
).stdout.strip()

if existing_hdfs:
    print(f"{HDFS_WORKDIR} sudah berisi data di HDFS, upload dilewati.")
else:
    print("Mengunggah dataset ke HDFS...")
    for class_folder in class_folders:
        subprocess.run(
            ["hdfs", "dfs", "-put", str(class_folder), f"{HDFS_WORKDIR}/"],
            check=True,
        )
    print("Upload dataset ke HDFS selesai.")

### **23. Verifikasi Dataset di HDFS**
Memeriksa hierarki file dan ruang penyimpanan yang digunakan dataset di HDFS.

In [ ]:
!hdfs dfs -ls {HDFS_WORKDIR}
!hdfs dfs -du -h {HDFS_WORKDIR}

### **24. Ringkasan Jumlah File dan Ukuran per Kelas**
Menghitung jumlah file fisik, total ukuran (MB), dan rasio ketimpangan (imbalance ratio) dataset di HDFS.

In [ ]:
def summarize_hdfs_classes(hdfs_root: str):
    listing = subprocess.run(
        ["hdfs", "dfs", "-ls", hdfs_root], capture_output=True, text=True
    ).stdout
    class_dirs = [line.split()[-1] for line in listing.splitlines() if line.startswith("d")]

    summary = []
    for class_dir in class_dirs:
        count_out = subprocess.run(
            ["hdfs", "dfs", "-count", class_dir], capture_output=True, text=True
        ).stdout.split()
        file_count = int(count_out[1])

        du_out = subprocess.run(
            ["hdfs", "dfs", "-du", "-s", class_dir], capture_output=True, text=True
        ).stdout.split()
        size_mb = int(du_out[0]) / (1024 * 1024)

        summary.append((class_dir.rstrip("/").split("/")[-1], file_count, size_mb))

    print(f"{'Kelas':<25}{'Jumlah File':>15}{'Ukuran (MB)':>15}")
    for name, count, size_mb in summary:
        print(f"{name:<25}{count:>15}{size_mb:>15.1f}")

    counts = [c for _, c, _ in summary]
    ratio = max(counts) / min(counts) if min(counts) > 0 else float("inf")
    print(f"\nRasio ketimpangan kelas (terbanyak / tersedikit): {ratio:.2f}")
    if ratio > 2.0:
        print("Perhatian: Terdapat ketimpangan kelas, pertimbangkan class weighting pada tahap modeling.")

    return summary

dataset_summary = summarize_hdfs_classes(HDFS_WORKDIR)

#### **24.1. Analisis Lanjutan Dataset (Distribusi Blok HDFS)**
Menghitung alokasi blok fisik HDFS (default 128 MB per block) untuk masing-masing kelas citra.

In [ ]:
print("=== Distribusi Blok HDFS per Kelas ===")
block_size_bytes = int(
    subprocess.run(
        ["hdfs", "getconf", "-confKey", "dfs.blocksize"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
)
print(f"Ukuran default blok HDFS: {block_size_bytes / (1024**2):.0f} MB")

def analyze_hdfs_partitions(hdfs_dir, block_size_bytes):
    result = subprocess.run(
        ["hdfs", "dfs", "-ls", "-R", hdfs_dir], capture_output=True, text=True, check=True
    )
    per_class_files, per_class_bytes, per_class_blocks = {}, {}, {}
    for line in result.stdout.splitlines():
        parts = line.split()
        if len(parts) < 8 or parts[0].startswith("d"):
            continue
        size_bytes = int(parts[4])
        class_name = Path(parts[-1]).parent.name
        num_blocks = max(1, -(-size_bytes // block_size_bytes))
        per_class_files[class_name] = per_class_files.get(class_name, 0) + 1
        per_class_bytes[class_name] = per_class_bytes.get(class_name, 0) + size_bytes
        per_class_blocks[class_name] = per_class_blocks.get(class_name, 0) + num_blocks
    return per_class_files, per_class_bytes, per_class_blocks

per_class_files, per_class_bytes, per_class_blocks = analyze_hdfs_partitions(
    HDFS_WORKDIR, block_size_bytes
)

print(f"{'Kelas':<25}{'File':>8}{'Ukuran (MB)':>15}{'Blok HDFS':>12}")
for class_name in CLASS_NAMES:
    files = per_class_files.get(class_name, 0)
    size_mb = per_class_bytes.get(class_name, 0) / (1024**2)
    blocks = per_class_blocks.get(class_name, 0)
    print(f"{class_name:<25}{files:>8}{size_mb:>15.1f}{blocks:>12}")

#### **24.2. Visualisasi Grafik Distribusi Data**
Menampilkan grafik batang jumlah file dan total ukuran data per kelas.

In [ ]:
import matplotlib.pyplot as plt

class_names_plot = [item[0] for item in dataset_summary]
file_counts_plot = [item[1] for item in dataset_summary]
sizes_mb_plot = [item[2] for item in dataset_summary]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot jumlah file per kelas
axes[0].bar(class_names_plot, file_counts_plot, color='steelblue')
axes[0].set_title('Distribusi Jumlah Citra per Kelas')
axes[0].set_xlabel('Kelas Diagnosis')
axes[0].set_ylabel('Jumlah Citra')
for idx, val in enumerate(file_counts_plot):
    axes[0].text(idx, val + max(file_counts_plot)*0.01, str(val), ha='center', va='bottom')

# Plot ukuran data per kelas (MB)
axes[1].bar(class_names_plot, sizes_mb_plot, color='coral')
axes[1].set_title('Distribusi Ukuran Data per Kelas (MB)')
axes[1].set_xlabel('Kelas Diagnosis')
axes[1].set_ylabel('Ukuran (MB)')
for idx, val in enumerate(sizes_mb_plot):
    axes[1].text(idx, val + max(sizes_mb_plot)*0.01, f"{val:.1f}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

#### **24.3. Visualisasi Contoh Sampel Citra Mentah per Kelas**
Menampilkan sampel citra dermatologi asli dari masing-masing kelas diagnosis untuk memverifikasi kondisi visual data.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

def display_sample_images_per_class(base_dir, class_names, num_samples=4):
    fig, axes = plt.subplots(len(class_names), num_samples, figsize=(num_samples * 3, len(class_names) * 3))
    if len(class_names) == 1:
        axes = axes.reshape(1, -1)

    for i, class_name in enumerate(class_names):
        class_path = base_dir / class_name
        images = list(class_path.glob("*.jpg")) + list(class_path.glob("*.png"))
        if not images:
            continue
        selected_images = random.sample(images, min(len(images), num_samples))
        for j, img_path in enumerate(selected_images):
            ax = axes[i, j]
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"{class_name}\n{img.size[0]}x{img.size[1]} px", fontsize=9)
            ax.axis('off')

    plt.tight_layout()
    plt.show()

display_sample_images_per_class(MERGED_DATASET_DIR, CLASS_NAMES, num_samples=4)

# **FASE 5 - Pipeline Preprocessing Terdistribusi (PySpark)**
Pada tahap **Data Preparation Bagian 1**, prapemrosesan citra dilakukan secara terdistribusi menggunakan PySpark RDD:
25. Inisialisasi PySpark Session dengan konfigurasi YARN
26. Konfigurasi Resolusi Preprocessing (224x224) dan Label Mapping
27. Fungsi Modular Preprocessing Terdistribusi via PyArrow & PIL
28. Jalankan Preprocessing Terdistribusi dan Catat Throughput
29. Pencatatan Durasi Preprocessing ke File Log `results.csv`
30. Verifikasi Sampel Hasil Preprocessing RDD (`.first()`)
31. Visualisasi Perbandingan Sebelum dan Sesudah Preprocessing (Raw vs Resized vs Histogram Normalisasi)

### **25. Inisialisasi PySpark Session**
Menghubungkan kernel notebook ke PySpark on YARN menggunakan pustaka `findspark`.

In [ ]:
import time
import csv

!pip install -q findspark pyarrow
import findspark

findspark.init(str(SPARK_HOME))

from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName(f"skincancer-preprocessing-{SCENARIO_NAME}")
    .getOrCreate()
)

print(f"Versi Spark: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")

### **26. Konfigurasi Preprocessing dan Label Mapping**
Menetapkan target resolusi citra standar model deep learning (224x224) serta pemetaan konsisten antara nama kelas dan indeks numerik.

In [ ]:
IMG_SIZE = 224

class_names = sorted(CLASS_NAMES)
label_to_index = {name: idx for idx, name in enumerate(class_names)}

print(f"Target Resolusi: {IMG_SIZE}x{IMG_SIZE} piksel (3 kanal RGB)")
print(f"Pemetaan Label ke Indeks: {label_to_index}")

### **27. Fungsi Modular Preprocessing Terdistribusi**
Membaca metadata path dari HDFS, membagi beban kerja ke `NUM_PARTITIONS` worker RDD, membaca byte stream via PyArrow, melakukan resize (224x224) dan normalisasi piksel [0.0, 1.0], kemudian menyimpan hasilnya di RDD cache (`persist(StorageLevel.DISK_ONLY)`).

In [ ]:
def run_distributed_preprocessing(spark, hdfs_dir, num_partitions, img_size, label_to_index):
    # 1. Ambil daftar path citra dari HDFS lewat CLI (hanya metadata path, ringan)
    listing = subprocess.run(
        ["hdfs", "dfs", "-ls", "-R", hdfs_dir],
        capture_output=True, text=True, check=True,
    ).stdout

    image_paths = []
    for line in listing.splitlines():
        parts = line.split()
        if len(parts) < 8 or parts[0].startswith("d"):
            continue
        path = parts[-1]
        if path.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(path)

    if not image_paths:
        raise RuntimeError(f"Tidak ada citra ditemukan di direktori HDFS: {hdfs_dir}")

    # 2. Distribusikan path ke RDD worker
    paths_rdd = spark.sparkContext.parallelize(image_paths).repartition(num_partitions)

    java_home = os.environ["JAVA_HOME"]
    hadoop_home = str(HADOOP_HOME)
    hadoop_classpath = subprocess.run(
        [f"{hadoop_home}/bin/hadoop", "classpath", "--glob"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()

    # 3. Eksekusi fungsi preprocessing per partisi di worker
    def preprocess_partition(paths):
        import os
        os.environ["JAVA_HOME"] = java_home
        os.environ["HADOOP_HOME"] = hadoop_home
        os.environ["LD_LIBRARY_PATH"] = f"{hadoop_home}/lib/native:{java_home}/lib/server"
        os.environ["CLASSPATH"] = hadoop_classpath

        import io
        import numpy as np
        from PIL import Image
        import pyarrow.fs as pafs

        hdfs = pafs.HadoopFileSystem("localhost", port=9000)

        for path in paths:
            try:
                with hdfs.open_input_stream(path) as f:
                    raw_bytes = f.read()

                img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
                img = img.resize((img_size, img_size))
                array = np.asarray(img, dtype=np.float32) / 255.0

                label_name = path.rstrip("/").split("/")[-2]
                label = label_to_index[label_name]

                image_id = Path(path).stem
                yield (array, label, image_id)
            except Exception as exc:
                print(f"Error memproses citra {path}: {exc}")

    processed_rdd = (
        paths_rdd
        .mapPartitions(preprocess_partition)
        .persist(StorageLevel.DISK_ONLY)
    )

    start_time = time.time()
    image_count = processed_rdd.count()
    elapsed_sec = time.time() - start_time

    partition_sizes = processed_rdd.glom().map(len).collect()
    print(f"Prapemrosesan terdistribusi selesai!")
    print(f"Jumlah partisi aktual: {len(partition_sizes)} (target: {num_partitions})")
    print(f"Distribusi citra per partisi: {partition_sizes}")

    return processed_rdd, elapsed_sec, image_count

### **28. Jalankan Preprocessing Terdistribusi dan Catat Throughput**
Mengeksekusi pipeline preprocessing pada HDFS dan mengukur durasi serta kecepatan proses (citra/detik).

In [ ]:
import psutil
import threading
import time

class ResourceMonitor:
    def __init__(self):
        self.keep_running = True
        self.cpu_records = []
        self.mem_records = []
        
    def monitor(self):
        while self.keep_running:
            self.cpu_records.append(psutil.cpu_percent(interval=None))
            self.mem_records.append(psutil.virtual_memory().used / (1024*1024))
            time.sleep(0.5)

monitor = ResourceMonitor()
psutil.cpu_percent(interval=None) # warmup
t = threading.Thread(target=monitor.monitor)
t.start()

processed_rdd, preprocessing_time_sec, image_count = run_distributed_preprocessing(
    spark, HDFS_WORKDIR, NUM_PARTITIONS, IMG_SIZE, label_to_index
)

monitor.keep_running = False
t.join()

avg_cpu = sum(monitor.cpu_records) / len(monitor.cpu_records) if monitor.cpu_records else 0
avg_mem = sum(monitor.mem_records) / len(monitor.mem_records) if monitor.mem_records else 0

throughput = image_count / preprocessing_time_sec if preprocessing_time_sec > 0 else 0
print(f"Total citra diproses : {image_count} citra")
print(f"Durasi prapemrosesan : {preprocessing_time_sec:.2f} detik")
print(f"Throughput komputasi : {throughput:.1f} citra/detik")
print(f"Rata-rata CPU Usage  : {avg_cpu:.2f}%")
print(f"Rata-rata RAM Usage  : {avg_mem:.2f} MB")

### **29. Pencatatan Durasi Preprocessing ke File Log `results.csv`**
Merekam metrik waktu prapemrosesan ke file `results.csv` sebagai dokumentasi kinerja komputasi.

In [ ]:
RESULTS_LOG_PATH = PROJECT_DIR / "results.csv"
LOG_COLUMNS = [
    "timestamp", "scenario", "num_workers", "num_partitions", "stage",
    "elapsed_sec", "image_count", "avg_cpu_percent", "avg_mem_mb",
    "accuracy", "f1_score", "loss", "roc_auc"
]

def log_preprocessing_time(scenario_name, num_workers, num_partitions, elapsed_sec, image_count, avg_cpu, avg_mem):
    file_exists = RESULTS_LOG_PATH.exists()
    with open(RESULTS_LOG_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(LOG_COLUMNS)
        writer.writerow([
            time.strftime("%Y-%m-%d %H:%M:%S"),
            scenario_name, num_workers, num_partitions,
            "preprocessing", f"{elapsed_sec:.2f}", image_count,
            f"{avg_cpu:.2f}", f"{avg_mem:.2f}",
            "", "", "", ""
        ])

log_preprocessing_time(SCENARIO_NAME, NUM_WORKERS, NUM_PARTITIONS, preprocessing_time_sec, image_count, avg_cpu, avg_mem)
print(f"Log preprocessing tersimpan di {RESULTS_LOG_PATH}")

### **30. Verifikasi Sampel Hasil Preprocessing RDD**
Mengambil sampel pertama dari RDD terdistribusi untuk memvalidasi dimensi array (224, 224, 3) dan tipe data nilai piksel.

In [ ]:
sample_array, sample_label, sample_id = processed_rdd.first()
print(f"Bentuk dimensi array (Shape): {sample_array.shape}")
print(f"Tipe data: {sample_array.dtype}")
print(f"Nilai piksel minimum: {sample_array.min():.4f}, maksimum: {sample_array.max():.4f}")
print(f"ID Citra: {sample_id}, Label numerik: {sample_label} ({class_names[sample_label]})")

assert sample_array.shape == (IMG_SIZE, IMG_SIZE, 3), "Dimensi citra tidak sesuai spesifikasi 224x224x3!"
assert 0.0 <= sample_array.min() and sample_array.max() <= 1.0, "Normalisasi piksel di luar rentang [0, 1]!"
print("Sanity check prapemrosesan berhasil 100%.")

### **31. Visualisasi Perbandingan Sebelum dan Sesudah Preprocessing**
Menampilkan perbandingan komparatif 3 kolom per sampel:
1. Citra Asli (ukuran piksel asli)
2. Citra Hasil Preprocessing (seragam 224x224)
3. Histogram Distribusi Piksel Ternormalisasi [0.0, 1.0]

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import io
import random

# Kumpulkan path citra per kelas dari HDFS untuk perbandingan
listing = subprocess.run(
    ["hdfs", "dfs", "-ls", "-R", HDFS_WORKDIR],
    capture_output=True, text=True, check=True
).stdout

paths_by_class = {c: [] for c in class_names}
for line in listing.splitlines():
    parts = line.split()
    if len(parts) > 0:
        path = parts[-1]
        if path.lower().endswith(('.jpg', '.jpeg', '.png')):
            label_name = path.rstrip("/").split("/")[-2]
            if label_name in paths_by_class:
                paths_by_class[label_name].append(path)

samples_per_class = 2
fig, axes = plt.subplots(len(class_names), samples_per_class * 3, figsize=(16, 3.5 * len(class_names)))
if len(class_names) == 1:
    axes = axes.reshape(1, -1)

for i, class_name in enumerate(class_names):
    sample_paths = random.sample(paths_by_class[class_name], min(len(paths_by_class[class_name]), samples_per_class))
    for j, sample_path in enumerate(sample_paths):
        cat_result = subprocess.run(
            ["hdfs", "dfs", "-cat", sample_path],
            capture_output=True, check=True
        )
        raw_bytes = cat_result.stdout
        original_img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        resized_img = original_img.resize((IMG_SIZE, IMG_SIZE))
        preprocessed_array = np.asarray(resized_img, dtype=np.float32) / 255.0

        col_base = j * 3
        # 1. Asli
        ax_orig = axes[i, col_base]
        ax_orig.imshow(original_img)
        ax_orig.set_title(f"1. Asli ({class_name})\n{original_img.size} px", fontsize=8)
        ax_orig.axis("off")

        # 2. Preprocessed
        ax_prep = axes[i, col_base + 1]
        ax_prep.imshow(preprocessed_array)
        ax_prep.set_title(f"2. Preprocessed\nShape: {preprocessed_array.shape}", fontsize=8)
        ax_prep.axis("off")

        # 3. Histogram
        ax_hist = axes[i, col_base + 2]
        ax_hist.hist(preprocessed_array.ravel(), bins=40, color='teal', alpha=0.7)
        ax_hist.set_title(f"3. Bukti Normalisasi\nMin: {preprocessed_array.min():.2f} | Max: {preprocessed_array.max():.2f}", fontsize=8)
        ax_hist.set_xlim(0, 1)
        ax_hist.set_yticks([])
        ax_hist.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
vis_path = PROJECT_DIR / f"preprocessing_norm_proof_{SCENARIO_NAME}.png"
plt.savefig(vis_path)
plt.show()
print(f"Grafik bukti normalisasi tersimpan di: {vis_path}")

# **FASE 6 - Data Splitting & Pemodelan**
Pada tahap **Data Preparation Bagian 2 & Modeling Bagian 1**:
32. Pembagian Data Train, Validation, dan Test (Stratifikasi Bebas Kebocoran)
33. Pengumpulan Data ke Driver dalam Format NumPy Array
34. Konstruksi Arsitektur Model Deep Learning (ResNet-50 + Triplet Attention - Model AK85)
35. Verifikasi Sanity Check Model (Forward Pass Dummy Batch & Model Summary)

### **32. Pembagian Data Train, Validation, dan Test Menggunakan CSV Split**
Pembagian data memanfaatkan partisi split yang telah kita susun secara metodologis pada tahap Data Understanding & Preparation sebelumnya:
- Pada **Skenario Irisan 3 Kelas**, pembagian membaca langsung file split `dataset_irisan_3kelas_train.csv`, `val.csv`, dan `test.csv` yang dibuat dengan `StratifiedGroupKFold` pada `lesion_id` sehingga **100% bebas kebocoran lesi pasien (*zero lesion leakage*)**.
- Pada **Skenario Biner**, pembagian dilakukan secara deterministik terstratifikasi per `image_id` dari CSV master.
Data uji (`test_rdd`) diisolasi penuh dan tidak akan pernah tersentuh pada tahap pelatihan.

In [ ]:
split_dir = csv_path.parent
train_csv_path = split_dir / "dataset_irisan_3kelas_train.csv"
val_csv_path = split_dir / "dataset_irisan_3kelas_val.csv"
test_csv_path = split_dir / "dataset_irisan_3kelas_test.csv"

if train_csv_path.exists() and val_csv_path.exists() and test_csv_path.exists():
    print("Memuat partisi split anti-kebocoran dari file CSV resmi:")
    df_train_csv = pd.read_csv(train_csv_path)
    df_val_csv = pd.read_csv(val_csv_path)
    df_test_csv = pd.read_csv(test_csv_path)

    train_ids = set(df_train_csv["image_id"].astype(str))
    val_ids = set(df_val_csv["image_id"].astype(str))
    test_ids = set(df_test_csv["image_id"].astype(str))

    print(f"  - Train Set : {len(train_ids)} citra")
    print(f"  - Val Set   : {len(val_ids)} citra")
    print(f"  - Test Set  : {len(test_ids)} citra (100% Bebas Kebocoran)")

    b_train = spark.sparkContext.broadcast(train_ids)
    b_val = spark.sparkContext.broadcast(val_ids)
    b_test = spark.sparkContext.broadcast(test_ids)

    train_rdd = processed_rdd.filter(lambda item: item[2] in b_train.value).map(lambda item: (item[0], item[1]))
    val_rdd = processed_rdd.filter(lambda item: item[2] in b_val.value).map(lambda item: (item[0], item[1]))
    test_rdd = processed_rdd.filter(lambda item: item[2] in b_test.value).map(lambda item: (item[0], item[1]))
else:
    raise FileNotFoundError("File split terpisah tidak ditemukan! Pastikan Anda sudah menjalankan Data Preparation Biner/Irisan sebelumnya.")

train_rdd = train_rdd.coalesce(NUM_PARTITIONS).persist(StorageLevel.DISK_ONLY)
val_rdd = val_rdd.persist(StorageLevel.DISK_ONLY)
test_rdd = test_rdd.persist(StorageLevel.DISK_ONLY)

def count_classes(rdd, num_classes):
    counts = rdd.map(lambda item: item[1]).countByValue()
    return [counts.get(i, 0) for i in range(num_classes)]

print("Distribusi kelas pada subset RDD:")
print(f"  - Train Set : {count_classes(train_rdd, NUM_CLASSES)} (Total: {train_rdd.count()})")
print(f"  - Val Set   : {count_classes(val_rdd, NUM_CLASSES)} (Total: {val_rdd.count()})")
print(f"  - Test Set  : {count_classes(test_rdd, NUM_CLASSES)} (Total: {test_rdd.count()})")

### **33. Pengumpulan Data ke Driver dalam Format NumPy Array**
Pada arsitektur Tipe 1, partisi data yang telah diproses dikumpulkan ke memori driver sebagai array NumPy. Subset uji (`X_test`, `y_test`) diisolasi secara ketat dan **TIDAK BOLEH** disentuh selama proses pelatihan!

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical

def rdd_to_numpy(rdd):
    arrays, labels = [], []
    for image_array, label in rdd.toLocalIterator():
        arrays.append(image_array)
        labels.append(label)
    return np.array(arrays, dtype=np.float32), np.array(labels, dtype=np.int32)

print("Mengumpulkan data Train, Val, dan Test ke memori driver...")
X_train, y_train = rdd_to_numpy(train_rdd)
X_val, y_val = rdd_to_numpy(val_rdd)
X_test, y_test = rdd_to_numpy(test_rdd)

y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)

print(f"Dimensi Train Set: X={X_train.shape}, y={y_train_cat.shape}")
print(f"Dimensi Val Set  : X={X_val.shape}, y={y_val_cat.shape}")
print(f"Dimensi Test Set : X={X_test.shape}, y={y_test_cat.shape}")

### **34. Arsitektur Model Deep Learning: ResNet-50 + Triplet Attention**
Membangun arsitektur model baseline AK85:
- **Backbone:** ResNet-50 pre-trained ImageNet (input 224x224x3).
- **Triplet Attention Module (WACV 2021):** Ditempatkan pada resolusi 14x14 (`conv4_block6_out`) dengan 3 branch interaktif: Channel-Height (C-H), Channel-Width (C-W), dan Spatial Attention (H-W).
- **Multi-Pooling Head:** Menggabungkan fitur Global Average Pooling, Global Max Pooling, dan Attended Features.
- **Classifier:** Batch Normalization, Dropout (0.5 & 0.4), Dense(512), Dense(256), dan Softmax Output.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
    BatchNormalization, Dropout, Concatenate, Input,
    Conv2D, Activation
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

# ---- MODUL TRIPLET ATTENTION ----
class ZPool(tf.keras.layers.Layer):
    def call(self, x):
        max_out = tf.reduce_max(x, axis=-1, keepdims=True)
        avg_out = tf.reduce_mean(x, axis=-1, keepdims=True)
        return tf.concat([max_out, avg_out], axis=-1)

class AttentionGate(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.zpool = ZPool()
        self.conv = Conv2D(1, kernel_size=3, padding='same', use_bias=False)
        self.sigmoid = Activation('sigmoid')

    def call(self, x):
        out = self.zpool(x)
        out = self.conv(out)
        out = self.sigmoid(out)
        return out

class TripletAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gate_ch = AttentionGate()
        self.gate_cw = AttentionGate()
        self.gate_hw = AttentionGate()

    def call(self, x):
        # Branch 1: C x H
        x_perm1 = tf.transpose(x, perm=[0, 3, 2, 1])
        attn1 = self.gate_ch(x_perm1)
        x_perm1 = x_perm1 * attn1
        x_perm1 = tf.transpose(x_perm1, perm=[0, 3, 2, 1])

        # Branch 2: C x W
        x_perm2 = tf.transpose(x, perm=[0, 1, 3, 2])
        attn2 = self.gate_cw(x_perm2)
        x_perm2 = x_perm2 * attn2
        x_perm2 = tf.transpose(x_perm2, perm=[0, 1, 3, 2])

        # Branch 3: H x W
        attn3 = self.gate_hw(x)
        x_branch3 = x * attn3

        return (x_perm1 + x_perm2 + x_branch3) / 3.0

# ---- PEMBANGUNAN MODEL LENGKAP ----
def build_ak85_model(img_size=IMG_SIZE, num_classes=3):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    base_model.trainable = False

    # Triplet Attention pada conv4 (14x14x1024)
    conv4_out = base_model.get_layer('conv4_block6_out').output
    attended = TripletAttention(name='triplet_attention')(conv4_out)
    attended_avg = GlobalAveragePooling2D(name='att_avg_pool')(attended)

    # Fitur utama ResNet50 (7x7x2048)
    x_main = base_model.output
    avg_pool = GlobalAveragePooling2D()(x_main)
    max_pool = GlobalMaxPooling2D()(x_main)

    # Penggabungan multi-scale pooling: (2048 + 2048 + 1024 = 5120 dimensi)
    x = Concatenate()([avg_pool, max_pool, attended_avg])
    x = BatchNormalization()(x)

    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.4)(x)

    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Fungsi pembangun model build_ak85_model() berhasil didefinisikan.")

### **35. Verifikasi Sanity Check Model**
Menguji forward pass model menggunakan batch dummy (4 citra ukuran 224x224) untuk memastikan dimensi output sesuai dengan jumlah kelas target.

In [ ]:
_test_model = build_ak85_model()
_dummy_batch = np.zeros((4, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
_dummy_output = _test_model.predict(_dummy_batch, verbose=0)

print(f"Output shape dummy batch size 4: {_dummy_output.shape}")
assert _dummy_output.shape == (4, 3), f"Shape tidak sesuai! Diharapkan (4, 3), didapat {_dummy_output.shape}"
print(f"Total parameter model: {_test_model.count_params():,}")
print("Sanity check forward pass model berhasil 100%.")

del _test_model, _dummy_batch, _dummy_output

# **FASE 7 - Pelatihan Terpusat GPU & Evaluasi Komprehensif**
Sesuai metodologi CRISP-DM pada tahap **Modeling Bagian 2 & Evaluation**:
36. Konfigurasi Pelatihan & Penanganan Ketimpangan Kelas (Class Weights)
37. Setup Callbacks Pelatihan (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
38. Eksekusi Pelatihan Terpusat pada GPU Driver (`model.fit`)
39. Visualisasi Kurva Pembelajaran (Loss & Akurasi: Train vs Validation)
40. Evaluasi Menyeluruh pada Test Set (Normalized Confusion Matrix & Classification Report)
41. Pencatatan Hasil Akhir ke File Log `results.csv`
42. Kesimpulan Data Scientist & Kesiapan Model

### **36. Konfigurasi Pelatihan & Penanganan Ketimpangan Kelas**
Menghitung bobot kelas (class weights) untuk mengatasi ketimpangan sebaran data citra secara otomatis.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

BATCH_SIZE = 32
print(f"Skenario: {SCENARIO_NAME} | Batch Size: {BATCH_SIZE} | Epochs: {EPOCHS}")

# Hitung class weights
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_arr))
print(f"Class Weights seimbang: {class_weights}")

### **37. Setup Callbacks Pelatihan**
Menyiapkan callback untuk adaptasi learning rate, early stopping jika konvergen, dan penyimpanan bobot model terbaik.

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

MODEL_SAVE_PATH = PROJECT_DIR / f"best_model_{SCENARIO_NAME}.h5"

callbacks = [
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_SAVE_PATH),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks siap dipasangkan ke pelatihan.")

### **38. Eksekusi Pelatihan Terpusat pada GPU Driver**
Melatih model ResNet-50 + Triplet Attention pada driver GPU menggunakan subset Train dan Validation saja. Data uji **TIDAK TERLIBAT** pada langkah ini.

In [ ]:
# Inisialisasi model final untuk pelatihan
model = build_ak85_model()

print("Memulai pelatihan model tersentralisasi pada GPU...")
train_start_time = time.time()

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

training_time_sec = time.time() - train_start_time
print(f"\nPelatihan GPU selesai dalam: {training_time_sec:.2f} detik ({training_time_sec/60:.2f} menit)")

### **39. Visualisasi Kurva Pembelajaran**
Menampilkan grafik perkembangan Loss dan Akurasi sepanjang epoch untuk mendeteksi konvergensi dan ada/tidaknya overfitting.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(13, 5))

# Grafik Akurasi
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o', color='royalblue')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s', color='darkorange')
plt.title(f'Kurva Akurasi - {SCENARIO_NAME}')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Grafik Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o', color='royalblue')
plt.plot(history.history['val_loss'], label='Val Loss', marker='s', color='crimson')
plt.title(f'Kurva Loss - {SCENARIO_NAME}')
plt.xlabel('Epoch')
plt.ylabel('Loss (Crossentropy)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
curve_path = PROJECT_DIR / f"training_curve_{SCENARIO_NAME}.png"
plt.savefig(curve_path)
plt.show()
print(f"Grafik pelatihan tersimpan di: {curve_path}")

### **40. Evaluasi Menyeluruh pada Test Set**
BARU PADA LANGKAH INI subset uji murni (`X_test`, `y_test`) dievaluasi:
- Menghitung Loss dan Akurasi Test
- Menghitung Macro F1-Score
- Menampilkan Normalized Confusion Matrix Heatmap
- Menampilkan Classification Report lengkap (Precision, Recall, F1-Score per kelas)

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_score

# 1. Evaluasi metrik dasar
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

macro_f1 = f1_score(y_test, y_pred, average='macro')
roc_auc = roc_auc_score(y_test_cat, y_pred_probs, multi_class='ovr')

print("=== HASIL EVALUASI TEST SET ===")
print(f"Test Loss       : {test_loss:.4f}")
print(f"Test Accuracy   : {test_accuracy * 100:.2f}%")
print(f"Test Macro F1   : {macro_f1:.4f}")
print(f"ROC-AUC (OVR)   : {roc_auc:.4f}")

# 2. Classification Report
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=class_names))

# 3. Normalized Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 5))
sns.heatmap(cm_normalized, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Normalized Confusion Matrix - {SCENARIO_NAME}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

### **41. Pencatatan Hasil Akhir ke File Log `results.csv`**
Merekam ringkasan performa pelatihan GPU dan metrik evaluasi test set ke file `results.csv` secara otomatis.

In [ ]:
def log_result(stage, elapsed_sec, image_count=None, accuracy=None, f1_score=None, loss=None, roc_auc=None, avg_cpu=None, avg_mem=None):
    file_exists = RESULTS_LOG_PATH.exists()
    with open(RESULTS_LOG_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(LOG_COLUMNS)
        writer.writerow([
            time.strftime("%Y-%m-%d %H:%M:%S"),
            SCENARIO_NAME, NUM_WORKERS, NUM_PARTITIONS,
            stage, f"{elapsed_sec:.2f}",
            image_count if image_count is not None else "",
            f"{avg_cpu:.2f}" if avg_cpu is not None else "",
            f"{avg_mem:.2f}" if avg_mem is not None else "",
            f"{accuracy:.4f}" if accuracy is not None else "",
            f"{f1_score:.4f}" if f1_score is not None else "",
            f"{loss:.4f}" if loss is not None else "",
            f"{roc_auc:.4f}" if roc_auc is not None else ""
        ])

log_result(
    stage="training_centralized_gpu",
    elapsed_sec=training_time_sec,
    image_count=len(X_train),
    accuracy=test_accuracy,
    f1_score=macro_f1,
    loss=test_loss,
    roc_auc=roc_auc
)
print(f"Log pelatihan tersimpan di {RESULTS_LOG_PATH}")

### **42. Kesimpulan Data Scientist & Kesiapan Model**
Ringkasan performa pipeline hybrid Spark on YARN Tipe 1:
- Prapemrosesan citra berhasil didistribusikan secara efisien melalui PySpark RDD pada klaster YARN.
- Arsitektur ResNet-50 dengan modul Triplet Attention pada conv4 (14x14) terbukti menangkap representasi spasial dan inter-channel lesi kulit dengan optimal.
- Seluruh tahapan berjalan sesuai standar CRISP-DM tanpa terjadinya kebocoran data (*data leakage*).